# Data Exploration — Phase 2

Raw scraping (Phase 1) is done: `data/raw/` has one parquet file per season, each with two rows per game (one per team). This notebook is for exploring that data before any feature-engineering or leakage decisions get made.

Two things this notebook deliberately keeps separate:
- **EDA** — what does the data actually look like (shape, coverage, balance, anomalies)?
- **Leakage/split planning** — what will be allowed to see what, once features exist?

Only the first is in scope here. Don't reach for anything that assumes a feature-engineering decision (e.g. a rolling average) — that belongs later.

Raw data is read-only — nothing in this notebook should write back to `data/raw/`.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Jupyter's cwd is this notebooks/ folder; add the project root so `src` is importable.
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## Load and combine the raw seasons

Each raw file is season-scoped and has no season identifier column of its own beyond `SEASON_ID` (a numeric code, not the `"2023-24"`-style string). Tag each file with its season string on load so it survives the concat.

In [3]:
raw_dir = PROJECT_ROOT / "data" / "raw"
season_files = sorted(raw_dir.glob("*.parquet"))

season_dfs = []
for file_path in season_files:
    season = file_path.stem.removesuffix("_data_raw")
    df = pd.read_parquet(file_path)
    df["SEASON"] = season
    season_dfs.append(df)

games = pd.concat(season_dfs, ignore_index=True)
games["GAME_DATE"] = pd.to_datetime(games["GAME_DATE"])

print(games.shape)
games.head()

(12300, 30)


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE,SEASON
0,22021,1610612744,GSW,Golden State Warriors,0022100002,2021-10-19,GSW @ LAL,W,240,41,...,50,30,9,2,17,18,121,7,1,2021-22
1,22021,1610612751,BKN,Brooklyn Nets,0022100001,2021-10-19,BKN @ MIL,L,240,37,...,44,19,3,9,13,17,104,-23,1,2021-22
2,22021,1610612749,MIL,Milwaukee Bucks,0022100001,2021-10-19,MIL vs. BKN,W,240,48,...,54,25,8,9,8,19,127,23,1,2021-22
3,22021,1610612747,LAL,Los Angeles Lakers,0022100002,2021-10-19,LAL vs. GSW,L,240,45,...,45,21,7,4,18,25,114,-7,1,2021-22
4,22021,1610612745,HOU,Houston Rockets,0022100008,2021-10-20,HOU @ MIN,L,240,40,...,41,21,13,3,24,25,106,-18,1,2021-22


## Shape & coverage

Does every season/team have the number of games you'd expect? Any gaps in `GAME_DATE` coverage worth explaining?

In [7]:
games.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12300 entries, 0 to 12299
Data columns (total 30 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   SEASON_ID          12300 non-null  object        
 1   TEAM_ID            12300 non-null  int64         
 2   TEAM_ABBREVIATION  12300 non-null  object        
 3   TEAM_NAME          12300 non-null  object        
 4   GAME_ID            12300 non-null  object        
 5   GAME_DATE          12300 non-null  datetime64[ns]
 6   MATCHUP            12300 non-null  object        
 7   WL                 12300 non-null  object        
 8   MIN                12300 non-null  int64         
 9   FGM                12300 non-null  int64         
 10  FGA                12300 non-null  int64         
 11  FG_PCT             12300 non-null  float64       
 12  FG3M               12300 non-null  int64         
 13  FG3A               12300 non-null  int64         
 14  FG3_PC

In [9]:
# Basic describe function

games.describe(include='all')

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE,SEASON
count,12300,1.230000e+04,12300,12300,12300,12300,12300,12300,12300.000000,12300.000000,...,12300.000000,12300.000000,12300.000000,12300.000000,12300.000000,12300.000000,12300.000000,12300.000000,12300.000000,12300
unique,5,NaN,30,30,6150,NaN,1740,2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
top,22021,NaN,GSW,Golden State Warriors,0022501194,NaN,NYK @ TOR,W,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021-22
freq,2460,NaN,410,410,2,NaN,11,6150,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2460
mean,NaN,1.610613e+09,NaN,NaN,NaN,2024-01-16 02:58:53.268292608,NaN,NaN,241.418699,41.684715,...,43.862276,25.985285,7.803984,4.844959,14.054228,19.361382,113.789187,0.000000,1.000163,NaN
min,NaN,1.610613e+09,NaN,NaN,NaN,2021-10-19 00:00:00,NaN,NaN,240.000000,22.000000,...,20.000000,8.000000,0.000000,0.000000,2.000000,4.000000,66.000000,-73.000000,1.000000,NaN
25%,NaN,1.610613e+09,NaN,NaN,NaN,2022-11-28 00:00:00,NaN,NaN,240.000000,38.000000,...,39.000000,22.000000,6.000000,3.000000,11.000000,17.000000,105.000000,-10.000000,1.000000,NaN
50%,NaN,1.610613e+09,NaN,NaN,NaN,2024-01-19 00:00:00,NaN,NaN,240.000000,42.000000,...,44.000000,26.000000,8.000000,5.000000,14.000000,19.000000,113.000000,0.000000,1.000000,NaN
75%,NaN,1.610613e+09,NaN,NaN,NaN,2025-03-05 00:00:00,NaN,NaN,240.000000,45.000000,...,48.000000,29.000000,10.000000,6.000000,17.000000,22.000000,122.000000,10.000000,1.000000,NaN
max,NaN,1.610613e+09,NaN,NaN,NaN,2026-04-12 00:00:00,NaN,NaN,315.000000,65.000000,...,74.000000,50.000000,22.000000,19.000000,31.000000,37.000000,176.000000,73.000000,2.000000,NaN


In [12]:
games.dtypes

SEASON_ID                    object
TEAM_ID                       int64
TEAM_ABBREVIATION            object
TEAM_NAME                    object
GAME_ID                      object
GAME_DATE            datetime64[ns]
MATCHUP                      object
WL                           object
MIN                           int64
FGM                           int64
FGA                           int64
FG_PCT                      float64
FG3M                          int64
FG3A                          int64
FG3_PCT                     float64
FTM                           int64
FTA                           int64
FT_PCT                      float64
OREB                          int64
DREB                          int64
REB                           int64
AST                           int64
STL                           int64
BLK                           int64
TOV                           int64
PF                            int64
PTS                           int64
PLUS_MINUS                  

In [ ]:
# Check if every team has all the number of games for the season - 82 games

games.groupby(['SEASON', 'TEAM_NAME'])['GAME_ID'].nunique().reset_index()

,SEASON,TEAM_NAME,GAME_ID
0,2021-22,Atlanta Hawks,82
1,2021-22,Boston Celtics,82
2,2021-22,Brooklyn Nets,82
3,2021-22,Charlotte Hornets,82
4,2021-22,Chicago Bulls,82
...,...,...,...
145,2025-26,Sacramento Kings,82
146,2025-26,San Antonio Spurs,82
147,2025-26,Toronto Raptors,82
148,2025-26,Utah Jazz,82


In [21]:
games.groupby(['SEASON', 'TEAM_NAME'])[['FGM', 'FGA', 'OREB', 'DREB', 'PTS', 'AST']].mean().reset_index().sort_values(by='PTS', ascending=False)

,SEASON,TEAM_NAME,FGM,FGA,OREB,DREB,PTS,AST
71,2023-24,Indiana Pacers,47.012195,92.670732,10.085366,31.439024,123.292683,30.756098
127,2025-26,Denver Nuggets,43.536585,87.756098,9.768293,34.207317,122.073171,28.963415
95,2024-25,Cleveland Cavaliers,44.536585,90.780488,11.182927,34.219512,121.939024,28.097561
104,2024-25,Memphis Grizzlies,44.756098,93.341463,12.878049,34.378049,121.707317,28.414634
135,2025-26,Miami Heat,43.682927,93.268293,11.829268,34.451220,120.865854,28.951220
...,...,...,...,...,...,...,...,...
92,2024-25,Brooklyn Nets,37.634146,86.097561,10.902439,30.402439,105.109756,25.158537
93,2024-25,Charlotte Hornets,38.317073,89.121951,12.231707,32.963415,105.097561,24.317073
8,2021-22,Detroit Pistons,38.158537,88.621951,10.975610,32.012195,104.829268,23.463415
21,2021-22,Orlando Magic,38.280488,88.292683,9.097561,35.182927,104.231707,23.743902


## Class balance (W/L)

The target is home vs. away win. `WL` here is per-team, not per-game — worth checking how home-court win rate looks once you can identify home vs. away (hint: `MATCHUP` contains `vs.` for home, `@` for away).

## Feature distributions

Pick a few raw stat columns (e.g. `PTS`, `FG_PCT`, `PLUS_MINUS`) and look at their distributions. Anything skewed, bimodal, or with suspicious outliers worth a second look?

## Anomalies & data quality follow-ups

The Phase 1 integrity check already flagged one missing `FT_PCT` value in `2023-24`. Worth a broader look here: any other nulls, impossible values (negative stats, `FG_PCT` outside [0, 1]), or team-name/ID inconsistencies across seasons (e.g. relocations, rebrands)?